In [197]:
from reedsolo import RSCodec
import base64
import json

In [198]:


class ReedSolomon:
    def __init__(self, ecc_symbols):
        self.ecc_symbols = ecc_symbols
        self.rsc = RSCodec(ecc_symbols)
    
    def encode(self, data: str, verbose: bool = False) -> bytearray:
        data_bytes = data.encode('utf-8')
        encoded = self.rsc.encode(data_bytes)
        if verbose:
            print(data, "- исходный текст")
            print(list(data_bytes), "- в байтах")
            print(list(encoded), "- код Рида-Соломона")
        return encoded
    
    def fix_errors(self, encoded_data: bytearray, verbose: bool = False) -> tuple[bytearray, int]:
        try:
            decoded, decoded_bytes, errata_pos = self.rsc.decode(encoded_data)
            errors_fixed = len(errata_pos) > 0
            encoded = self.rsc.encode(decoded)
            if verbose:
                print(list(encoded_data), "- до исправления ошибок")
                print(list(encoded), "- после исправления ошибок")
            return encoded, errors_fixed
            
        except Exception:
            return None, False, -1
    
    def decode(self, encoded_data: bytearray, verbose: bool = False) -> str:
        decoded, *_ = self.rsc.decode(encoded_data)
        decoded_str = decoded.decode('utf-8')
        if verbose:
            print(list(encoded_data), "- код Рида-Соломона")
            print(list(decoded), "- декодированная строка")
            print(decoded_str, "- перевели обратно в текст")
        return decoded_str


    def simulate_errors(self, encoded_data: bytearray, error_positions: list[int], verbose: bool = False) -> bytearray:
        corrupted = bytearray(encoded_data)
        
        for pos in error_positions:
            if pos < len(corrupted):
                # Инвертируем бит в указанной позиции
                corrupted[pos] ^= 0xFF  # Меняем все биты байта
        
        broken_data = bytearray(corrupted)
        if verbose:
            print(list(encoded_data), "- код Рида-Соломона")
            print(list(broken_data), "- поломанный код Рида-Соломона")
        return broken_data


In [199]:
rs = ReedSolomon(ecc_symbols=2)  # ecc_symbols//2 ошибок

In [203]:
original_text = "hello"
encoded = rs.encode(original_text, verbose=True)

hello - исходный текст
[104, 101, 108, 108, 111] - в байтах
[104, 101, 108, 108, 111, 236, 142] - код Рида-Соломона


In [204]:
encoded = rs.simulate_errors(encoded, error_positions=[0], verbose=True)

[104, 101, 108, 108, 111, 236, 142] - код Рида-Соломона
[151, 101, 108, 108, 111, 236, 142] - поломанный код Рида-Соломона


In [205]:
encoded, errors_fixed = rs.fix_errors(encoded, verbose=True)

[151, 101, 108, 108, 111, 236, 142] - до исправления ошибок
[104, 101, 108, 108, 111, 236, 142] - после исправления ошибок


In [206]:
decoded = rs.decode(encoded, verbose=True)

[104, 101, 108, 108, 111, 236, 142] - код Рида-Соломона
[104, 101, 108, 108, 111] - декодированная строка
hello - перевели обратно в текст
